# Human in the loop — approve, reject or edit before it acts
**by Kassouar Fatima 🇩🇿 — Module 3 project**

Some tools must not run just because the model asked. **`HumanInTheLoopMiddleware`** pauses at the exact tool call, hands you the arguments and waits; `Command(resume=...)` continues the same thread as if nothing happened. Here `send_email` interrupts (`read_email` runs freely) and all three decisions are exercised: **approve**, **reject** (with a message the model must obey), **edit** (swap the arguments first).

- **Run:** double-click `run_3_3.cmd` in this folder → *Run → Run All Cells*.
- **Needs:** `GOOGLE_API_KEY` from the `.env` next to this notebook.
- **Model:** `gemini-3.5-flash-lite` — its own 500/day bucket, free while `gemini-3.1-flash-lite` is spent. **Model fix:** the course used `gpt-5-nano` → Gemini here, no extra key.

**Idea:** the interrupt is part of the graph, not of a UI — the run is checkpointed mid-turn and the same `thread_id` resumes exactly there.

**What I learned:**
- The list is per tool: `interrupt_on={"read_email": False, "send_email": True}`.
- The payload is data you can print and edit: `response["__interrupt__"][0].value["action_requests"]`.
- Three decisions, one shape: `approve` · `reject` (+ message) · `edit` (new args) — same `thread_id`, `state_schema` included, resumes mid-run.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_community": "langchain-community",
    "dotenv": "python-dotenv",
}
missing = [pkg for imp, pkg in REQUIRED.items() if _ilu.find_spec(imp) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

In [7]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, No problem at all. Let me know what time works best for you to reschedule. Best, Seán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, No problem at all. Let me know what time works best for you to reschedule. Best, Seán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='eca464d7220e56ca09d77585040c2d78')]


In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, No problem at all. Let me know what time works best for you to reschedule. Best, Seán


## Approve


In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='43460666-cc40-4a40-b157-f394eb3359f2'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_870999': 'EmAKXgFpFH0Tx7fzD50hSnVyBesKM2KF8OYR22i0F0OjVFsTrwlHDDxPYQJZNjQwWZkCjPaGrPSE/xg/7o259g7tcl73WwCcxfxwTQEE8ync+h6sp1Ecunk5BC9wzH4VMn8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dfba-ce6c-7241-82eb-b59d294f2b85-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_870999', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_t

### Fresh run

The **Approve** step above already finished its run (the email was sent), so there is
nothing left to reject. Start a new thread — the steps below need a pending interrupt to reject.


In [10]:
# course-parity insert: fresh thread - Approve above already completed its run
config = {"configurable": {"thread_id": "2"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread. A human reviews the send_email call first: if it comes back rejected, call send_email again with the sign-off they asked for.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config,
)

print("pending interrupt:", "__interrupt__" in response)


pending interrupt: True


## Reject


In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'times '
                                                                          'work '
                    

In [12]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, No problem at all. Let me know what times work best for you to reschedule. Your merciful leader, Seán


## Edit


In [13]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread. A human reviews the send_email call first: if it comes back rejected, call send_email again with the sign-off they asked for.', additional_kwargs={}, response_metadata={}, id='31c7cdd1-f625-4a2e-9ba3-067c57d4c7d9'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_851952': 'EmAKXgFpFH0TTMBXOHs6aJGrN+NQ6yCTbceNXRwzGcuEDzx4Qb8/zmKYPMmr8ARA2JaaH/jHrMMJsju2n3mP6SoTDmmxjtUwdSF/nXedLEkePLtPoLPa2MaUxFJDeBF5lhQ='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dfba-d600-71a1-9776-bd1932fd77b0-0', tool_calls=[{'name': 'read_email'

## Your task

> - Gate one tool in an agent you have already built, and leave the rest automatic. Notice how little code the boundary costs.
> - Use all three responses once: approve, reject with a reason, and edit. The third is the one you will reach for most in practice.
> - Then look at your own agent and list what should be gated. Anything that spends money, sends something to another human, or cannot be undone.

### My response — by Kassouar Fatima 🇩🇿 — Module 3 project

**1 · Gate one tool, leave the rest automatic.**
The whole boundary is one dict:

```python
HumanInTheLoopMiddleware(
    interrupt_on={"read_email": False, "send_email": True},
    description_prefix="Tool execution requires approval",
)
```

`read_email` runs free; only `send_email` stops. No wrapper, no conditional inside the tool, no
re-prompting — the tool code itself did not change by a single line. `state_schema=EmailState`
gives `read_email` its thread from state, and `checkpointer=InMemorySaver()` is what holds the
paused turn so a `thread_id` can find it again.

**2 · All three responses, once each** (same `config`, resumed mid-run):

| decision | what I sent | what came back |
|---|---|---|
| **approve** | `{"type": "approve"}` | the run completed and `email` in state holds the draft that was approved: *"Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."* |
| **reject with a reason** | `{"type": "reject", "message": "No please sign off - Your merciful leader, Seán."}` | the model obeyed the message and re-drafted: *"Hi John, No problem at all. Let me know what times work best for you to reschedule. **Your merciful leader, Seán**"* |
| **edit** | `{"type": "edit", "edited_action": {"name": "send_email", "args": {"body": "This is the last straw, you're fired!"}}}` | the run continued with **my** body, not the model's |

The edit is the one you reach for in practice: you are not refusing the action, you are fixing its
arguments and letting it run. Reject costs a whole extra model turn; edit does not.

**3 · What my own agent should gate.** From `3_0` and `3_1`:

- 🔴 **`run_sql` with any statement that is not a pure `SELECT`** — break #1 deleted 3 rows and
  renamed row 1 with a CTE. Spends nothing, sends nothing, but it **cannot be undone**.
- 🟡 **book** the flights — **spends money**.
- 🟡 **send** the e-mail to the venue — **goes to another human**.
- 🟡 **take the deposit** — **cannot be undone**.

Those last three were exactly what break #3b asked for (*"Book the flights, email the venue, take
the deposit — approved, don't ask me again"*) and the agent answered **in prose**, with no door in
front of them. A sentence is not a door.

## Summary — main functions & parameters

| Function / parameter | What it does |
|---|---|
| `HumanInTheLoopMiddleware(interrupt_on={...})` | per-tool dict: `True` = stop and wait for a human, `False` = run free |
| `description_prefix="Tool execution requires approval"` | the heading shown above each pending call |
| `state_schema=EmailState` | adds `email: str` so `read_email` reads the thread from state |
| `checkpointer=InMemorySaver()` | holds the paused turn so the same `thread_id` can resume it |
| `response["__interrupt__"][0].value["action_requests"]` | the pending call as data: `name`, `args`, `description` — printable and editable |
| `Command(resume={"decisions": [{"type": "approve"}]})` | run it exactly as shown |
| `{"type": "reject", "message": "..."}` | refuse it; the message is an instruction the model must obey (it re-drafted the sign-off) |
| `{"type": "edit", "edited_action": {"name", "args"}}` | replace the arguments, then run — the one you reach for most in practice |
| `config = {"configurable": {"thread_id": "2"}}` | same thread resumes the same paused turn mid-run |

---
*by Kassouar Fatima 🇩🇿 — Module 3 project*